# Problem 021:  Amicable Numbers

> Let $d(n)$ be defined as the sum of proper divisors of $n$ (numbers less than $n$ which divide evenly into $n$).
>
> If $d(a) = b$ and $d(b) = a$, where $a \ne b$, then $a$ and $b$ are an amicable pair and each of $a$ and $b$ are called amicable numbers.
>
> For example, the proper divisors of $220$ are $1, 2, 4, 5, 10, 11, 20, 22, 44, 55$ and $110$; therefore $d(220) = 284$. The proper divisors of $284$ are $1, 2, 4, 71$ and $142$; so $d(284) = 220$.
>
> Evaluate the sum of all the amicable numbers under $10000$.


## Solution $\mathcal O (N\ln N \ln\ln N)$

The only way to find these values, that I can think of, is to actaully find the sum of proper divisors for every value up to $N$ and then going through the list and finding the ones with the desired property.

One immediate problem is when $d(n) > N$, as in one of the amicable pair could be less than the limit and the other greater than the limit.  To make sure these situations are accounted for, the sum of proper divisors need to carried out to larger values.  With a little digging through the Wikipedia page for the [Divisor Function](https://en.wikipedia.org/wiki/Divisor_function), the following relation is found for an upper bound on $d(n)$,
$$d(n) < e^\gamma n \ln \ln n + \frac{0.6483n}{\ln\ln n},$$
where $\gamma$ is the Euler-Mascheroni constant.  So to catch any amicable pairs that split $N$, the sum of proper divisors up to that limit $M$ needs to be considered.

There are a few ways to find the sum of proper divisors for a range of values.  One way is to do it like a sieve.  In this approach, a loop is done for all factors $i$ less than or equal to $M/2$ and added to every $i^\mbox{th}$ element from $2i$ up to $M$.  This algorithm will take $M/i$ steps for every $i$, making it $\mathcal O(M\ln M)$.  Given the above relation between $N$ and the maximum value that needs to be found as $M \sim N\ln\ln N$, that makes the overall algorithm $\mathcal O(N\ln N\ln\ln N)$.

In [131]:
%%time

from math import log

def p021_try1(N: int) -> int:
    M = int(1.781072418 * N * log(log(N)) + 0.6483 * N / log(log(N))) + 1 - N
    N += 1
    a = [1] * M
    imax = M // 2
    for i in range(2, imax):
        a[2*i::i] = [i + anow for anow in a[2*i::i]]
    return sum([i for i in range(N) if a[i] != i and a[a[i]] == i])
    
N = 10_000
ans = p021_try2(N)

print(ans)

31626
CPU times: user 23.9 ms, sys: 11 μs, total: 23.9 ms
Wall time: 23.1 ms


## A faster solution, also $\mathcal O (N \ln N \ln \ln N)$

Here is another tack for finding the sum of proper divisors based on the prime factorization of a number.  The seive of Eratosthenes can be changed slightly to produce a list of the smallest prime factor of every integer up to $N$.  The array of smallest prime factors can then be used to find the prime factorization of $n$ in on average $\ln n$ time.

Given that a number $n$ is written in terms of its prime factorization as
$$n = \prod_i p_i^{n_i},$$
the sum of all the divisors, including $n$ itself, can be written as
$$d(n)+n = \prod_i \sum_{k=0}^{n_i} p^k = \prod_i \frac{p_i^{n_i+1} - 1}{p_i - 1}.$$

An outline of this algorithm is as follows.  The smallest prime factors of all values up to $M$ are first found.  For all values less than or equal to $N$, the sum of proper divisors is found from the prime factorization of each value.  Then if the value has a sum of proper divisors greater than itself, the sum of proper divisors of that sum is found.  This will require a new calculation if the first sum is greater than $N$.  Then if the new sum is equal to the original value in question, an amicable pair has been found.

The overall time complexity for this algorithm is also $\mathcal O(N \ln N \ln \ln N)$, due to the seiving algorithm up to $M$.  In practice, this second approach is much faster.  The key is twofold. The sieve in the second approach is a lot more efficient than the one in the first approach, since you are updating only values for prime factors and at indices greater than $p^2$.  Also, you do not need to find the sum of proper divisors for _all_ values up to $M$, so the first algorithm spends a lot of time finding all those values whereas the second one calculates only the ones needed.

In [132]:
%%time

from math import log

def sieve_of_factors(num: int) -> list[int]:
    """ Returns an array of smallest prime factors using a seive.
    """
    sieve = [None] * ((num-1) // 2)
    pmax = int(num**0.5)//2
    for p in range(pmax):
        if sieve[p] is None:
            k = 2 * p + 3
            sieve[p] = k
            sieve[k**2//2-1::k] = [k if s is None else s for s in sieve[k**2//2-1::k]]
    return [2*i+3 if s is None else s for i, s in enumerate(sieve)]


def sum_of_divisors(x: int, s: list[int]):
    powp = 2
    while x % 2 == 0:
        x //= 2
        powp *= 2
    val = powp - 1
    while x != 1:
        p = s[x//2-1]
        powp = p * p
        x //= p
        while x != 1 and s[x//2-1] == p:
            powp *= p
            x //= p
        val *= (powp - 1) // (p - 1)
    
    return val
    

def p021_try2(N: int) -> int:
    M = int(1.781072418 * N * log(log(N)) + 0.6483 * N / log(log(N))) + 1
    s = sieve_of_factors(M)

    val = 0
    sopd = [0] + [sum_of_divisors(i, s) - i for i in range(1,N+1)]
    for i in range(1, N):
        snow = sopd[i]
        if snow > i:
            if snow <= N:
                if sopd[snow] == i:
                    val += i + snow
            else:
                if i == sum_of_divisors(snow, s) - snow:
                    val += i
    
    return val
        
N = 10_000
ans = p021_try2(N)

print(ans)

31626
CPU times: user 19.3 ms, sys: 0 ns, total: 19.3 ms
Wall time: 18.8 ms
